# 항공기 도착 지연 예측

**목표:** 예정 출발 시점에 이미 알 수 있는 정보와 출발 공항의 날씨를 사용해 `arr_delay`(도착 지연 시간, 분)를 예측한다.

> 실행 전 `xgboost`, `scikit-learn`이 없다면 첫 코드 셀의 설치 줄 주석을 해제해 한 번만 실행한다.

In [ ]:
# 필요한 패키지가 처음이라면 한 번만 실행
# %pip install xgboost scikit-learn

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

RANDOM_STATE = 42
pd.set_option('display.max_columns', 100)

## 1. 데이터 불러오기

수업에서 사용한 `Path("./data/파일명.csv")` 방식으로 각 CSV 파일의 경로를 명시한다.

In [ ]:
# 수업의 Path 사용 방식: 파일 경로를 명시적으로 작성한다.
# 폴더명은 실제 파일 시스템에 저장된 한글 자모 형태를 그대로 사용한다.
DATA_DIR = Path('./data') / '\u1106\u1175\u1109\u1175\u11ab\u1105\u1165\u1102\u1175\u11bc\u1111\u1167\u11bc\u1100\u1161_\u1103\u1166\u110b\u1175\u1110\u1165'
FLIGHTS_PATH = DATA_DIR / 'flights.csv'
AIRPORTS_PATH = DATA_DIR / 'airports.csv'
WEATHER_PATH = DATA_DIR / 'weather.csv'
AIRLINES_PATH = DATA_DIR / 'airlines.csv'
PLANES_PATH = DATA_DIR / 'planes.csv'
TZONE_META_PATH = DATA_DIR / 'tzone_meta.csv'

flights = pd.read_csv(FLIGHTS_PATH)
airports = pd.read_csv(AIRPORTS_PATH)
weather = pd.read_csv(WEATHER_PATH)
airlines = pd.read_csv(AIRLINES_PATH)
planes = pd.read_csv(PLANES_PATH)
tzone_meta = pd.read_csv(TZONE_META_PATH)

print('flights :', flights.shape)
print('airports:', airports.shape)
print('weather :', weather.shape)
display(flights.head())
display(weather.head())

## 2. 날씨 데이터 결측치 확인

`isna().sum()`으로 컬럼별 결측치를 세고, 0보다 큰 컬럼만 출력한다.

In [ ]:
weather_missing = weather.isna().sum()
weather_missing = weather_missing[weather_missing > 0].sort_values(ascending=False)
print(weather_missing)

## 3. 공항별 시간 순서 보간

공항별 관측치를 시간순으로 정렬한 뒤 수치형 날씨 컬럼만 `method='time'`으로 보간한다. 서로 다른 공항의 값을 섞어 보간하면 안 되므로 반드시 `origin`별로 처리한다.

In [ ]:
weather = weather.copy()
weather['time_hour'] = pd.to_datetime(weather['time_hour'])
weather = weather.sort_values(['origin', 'time_hour'])

# 날짜/시간 키는 보간 대상이 아니며, 실제 기상 관측값만 대상으로 한다.
weather_value_cols = [
    'temp', 'dewp', 'humid', 'wind_dir', 'wind_speed',
    'wind_gust', 'precip', 'pressure', 'visib'
]

# time_hour를 인덱스로 두고 공항(origin)별로만 시간 보간한다.
timed_weather = weather.set_index('time_hour')
interpolated_values = (
    timed_weather.groupby('origin')[weather_value_cols]
    .transform(lambda group: group.interpolate(method='time', limit_direction='both'))
)
weather[weather_value_cols] = interpolated_values.to_numpy()

print('보간 후 날씨 결측치')
print(weather[weather_value_cols].isna().sum())

## 4. 공항/날씨 병합

수업에서 다룬 병합 관점에서 `concat`은 같은 형태의 표를 위아래로 붙일 때 사용하고, `merge`는 공통 키를 기준으로 열을 연결할 때 사용한다. 여기서는 항공편 한 행을 보존해야 하므로 `pd.merge(..., how='left')`가 적합하다.

날씨의 고유 키는 `공항 + 연/월/일/시`이다. `time_hour`는 표준시 표현 차이가 있을 수 있어 병합 키로 쓰지 않는다.

In [ ]:
# 병합할 키의 자료형을 맞춘다.
for col in ['year', 'month', 'day', 'hour']:
    weather[col] = weather[col].astype(int)
    flights[col] = flights[col].astype(int)

# (1) 출발 공항 날씨: origin + 연/월/일/시
weather_for_dep = weather.drop(columns='time_hour').rename(
    columns={col: f'dep_weather_{col}' for col in weather_value_cols}
)
flights_merged = flights.merge(
    weather_for_dep,
    on=['origin', 'year', 'month', 'day', 'hour'],
    how='left',
    validate='many_to_one'
)

# (2) 도착 공항 날씨: dest + 연/월/일/시
weather_for_arr = weather.drop(columns='time_hour').rename(
    columns={'origin': 'dest', **{col: f'arr_weather_{col}' for col in weather_value_cols}}
)
flights_merged = flights_merged.merge(
    weather_for_arr,
    on=['dest', 'year', 'month', 'day', 'hour'],
    how='left',
    validate='many_to_one'
)

# 공항 메타데이터도 출발/도착 공항별로 붙인다.
airport_cols = ['faa', 'lat', 'lon', 'alt', 'tz']
origin_airports = airports[airport_cols].rename(
    columns={'faa': 'origin', 'lat': 'origin_lat', 'lon': 'origin_lon',
             'alt': 'origin_alt', 'tz': 'origin_tz'}
)
dest_airports = airports[airport_cols].rename(
    columns={'faa': 'dest', 'lat': 'dest_lat', 'lon': 'dest_lon',
             'alt': 'dest_alt', 'tz': 'dest_tz'}
)
flights_merged = flights_merged.merge(origin_airports, on='origin', how='left')
flights_merged = flights_merged.merge(dest_airports, on='dest', how='left')

print('병합 후 행/열 수:', flights_merged.shape)
print('출발지 날씨 매칭률:', flights_merged['dep_weather_temp'].notna().mean())
print('도착지 날씨 매칭률:', flights_merged['arr_weather_temp'].notna().mean())

### 도착지 날씨에 대한 데이터 제약

이 데이터의 날씨 관측 공항은 EWR, JFK, LGA뿐인데 `flights.dest`에는 이 공항들이 없다. 따라서 위 `left merge`는 요구사항대로 수행되지만 `arr_weather_*`는 전부 결측이다. 없는 관측값을 다른 도시의 날씨로 보간하면 잘못된 데이터가 되므로, 모델 입력에서는 제거한다. 실제 도착지 기상 데이터가 추가되면 같은 병합 코드로 바로 사용할 수 있다.

## 5-6. 타깃과 입력 변수 구성

- 타깃은 `arr_delay`이다. 도착 예정 시각 대비 실제 도착 지연 **분**인 연속형 수치이므로 회귀 문제이며 `XGBRegressor`와 맞는다.
- 실제 도착 결과(`arr_time`, `air_time`)와 타깃 자체는 정답을 미리 알려주는 누수이므로 제거한다.
- 예정 출발 시점 예측을 가정하므로 실제 출발 결과(`dep_time`, `dep_delay`)도 제거한다.
- `year`는 모든 행이 2017년인 상수 컬럼이고, `flight`와 `tailnum`은 단순 식별자이며, `time_hour` 원본 문자열의 정보는 이미 연·월·일·시 컬럼에 있으므로 제거한다.
- 도착지 날씨 컬럼은 이 데이터에서는 전부 결측이므로 제거한다.

In [ ]:
# 타깃이 없는 행은 학습 정답이 없으므로 제외한다.
model_df = flights_merged.dropna(subset=['arr_delay']).copy()

# 같은 날짜·시간 안에서도 예정 출발 분(minute) 순서를 보존하기 위한 시각 변수다.
model_df['scheduled_datetime'] = (
    pd.to_datetime(model_df['time_hour'])
    + pd.to_timedelta(model_df['minute'], unit='min')
)

target = 'arr_delay'
leakage_or_unnecessary_cols = [
    target,                 # 타깃은 X에 포함하면 안 됨
    'arr_time', 'air_time', # 실제 도착 후에야 알 수 있는 값
    'dep_time', 'dep_delay',# 예정 출발 시점에는 알 수 없는 실제 출발 결과
    'year',                 # 모든 행이 2017인 상수 컬럼
    'flight', 'tailnum',    # 식별자 성격의 고유/고카디널리티 값
    'time_hour',            # year/month/day/hour로 이미 분리된 원본 시간 문자열
    'scheduled_datetime',   # 시간순 분할에만 사용한 정렬 기준
] + [f'arr_weather_{col}' for col in weather_value_cols]

cols_to_drop = [col for col in leakage_or_unnecessary_cols if col in model_df.columns]
print('제거한 컬럼:', cols_to_drop)

## 7-8. 학습/테스트 분리 및 XGBoost 회귀 모델 학습

수업의 자전거 수요 예제처럼 시간 데이터는 무작위로 섞지 않는다. 과거 80%를 학습에, 이후 20%를 테스트에 사용해 미래 정보가 학습 데이터에 섞이는 것을 막는다.

In [ ]:
# 시간순 정렬 후 앞 80%는 학습, 뒤 20%는 테스트로 사용한다.
model_df = model_df.sort_values('scheduled_datetime').reset_index(drop=True)

X = model_df.drop(columns=cols_to_drop)
y = model_df[target]

split_index = int(len(model_df) * 0.8)
X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# 학습·테스트를 분리한 후 범주형 문자열을 0/1 더미 변수로 변환한다.
X_train = pd.get_dummies(X_train, columns=X_train.select_dtypes(include='object').columns, dummy_na=True)
X_test = pd.get_dummies(X_test, columns=X_test.select_dtypes(include='object').columns, dummy_na=True)
# 테스트에만 등장한 범주는 새 입력 컬럼으로 쓰지 않고, 학습 컬럼 구조에 맞춘다.
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print('학습 데이터:', X_train.shape, y_train.shape)
print('테스트 데이터:', X_test.shape, y_test.shape)
print('학습 기간:', model_df.loc[:split_index - 1, 'scheduled_datetime'].min(), '~',
      model_df.loc[:split_index - 1, 'scheduled_datetime'].max())
print('테스트 기간:', model_df.loc[split_index:, 'scheduled_datetime'].min(), '~',
      model_df.loc[split_index:, 'scheduled_datetime'].max())

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('학습 완료')

## 9. 테스트 예측과 회귀 평가

MAE는 평균적인 분 단위 오차, RMSE는 큰 지연 오차에 더 민감한 분 단위 오차, R²는 지연 시간 변화의 설명 정도다. MAE와 RMSE가 크게 차이나면 일부 항공편의 큰 지연을 특히 잘 맞히지 못한다는 뜻이다.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

metrics_df = pd.DataFrame([{
    'MAE': mae,
    'MSE': mse,
    'RMSE': rmse,
    'R2': r2
}])
display(metrics_df.round(3))

print(f'평균 절대 오차는 약 {mae:.1f}분입니다.')
if rmse > mae * 1.3:
    print('RMSE가 MAE보다 커 일부 항공편의 큰 지연이 예측 오차를 키우고 있습니다.')
else:
    print('MAE와 RMSE의 차이가 비교적 작아 큰 오차의 영향이 상대적으로 크지 않습니다.')
if r2 < 0:
    print('R²가 0보다 작아 평균 지연시간만 예측하는 기준보다 성능이 낮습니다.')
elif r2 < 0.3:
    print('R²가 높지 않아 날씨·일정 외의 지연 요인도 많이 작용함을 시사합니다.')
else:
    print('R²가 양수이며, 입력 변수들이 도착 지연의 일부 변동을 설명합니다.')

## 10. Feature Importance

중요도는 모델이 분할에 활용한 상대적 정도이며, 인과관계를 뜻하지는 않는다. 상위 2개 특성을 출력하고 막대그래프로 확인한다.

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

print('가장 중요한 Feature 2개')
print(importance.head(2))

plt.figure(figsize=(10, 6))
importance.head(20).sort_values().plot(kind='barh', color='steelblue')
plt.title('XGBoost Feature Importance (Top 20)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()